# Notebook 01: PrecipitationAFNO Baseline — Stage 2 Evaluation

Compare ERA5 coarse input (0.25°) vs PrecipitationAFNO output over Singapore.  
Stage 2 produced 10 hindcast timestamps across 3 known heavy-rain events.

> **Note:** CorrDiff NIM was blocked (self-hosted only, Taiwan domain mismatch).  
> Replaced with PrecipitationAFNO via WB2ERA5 (global ERA5 from WeatherBench2 / GCS).  
> Output is 0.25° — a coarse baseline, **not** the 1–2 km primary deliverable.

**Prerequisites:**
- Stage 1 complete: `data/raw/era5/singapore_2022.zarr` exists
- Stage 2 complete: `data/stage2/precip_*.nc` files exist (10 files)

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd

STAGE2_DIR = ROOT / "data" / "stage2"
ERA5_PATH  = ROOT / "data" / "raw" / "era5" / "singapore_2022.zarr"
RESULTS    = ROOT / "results"
RESULTS.mkdir(exist_ok=True)

# Singapore bbox (matches run_precipitation_afno.py)
SG_LAT_MIN, SG_LAT_MAX = 0.8, 1.6
SG_LON_MIN, SG_LON_MAX = 103.4, 104.2

nc_files = sorted(STAGE2_DIR.glob("precip_*.nc"))
print(f"Stage 2 files found: {len(nc_files)}")
for f in nc_files:
    print(f"  {f.name}")

In [ ]:
# Load all Stage 2 AFNO outputs as a single multi-time dataset
afno_ds = xr.open_mfdataset(nc_files, combine="nested", concat_dim="time")
afno_ds = afno_ds.sortby("time")

print(afno_ds)
print(f"\ntp range: {float(afno_ds.tp.min()):.3f} – {float(afno_ds.tp.max()):.3f} mm/6h")
print(f"Events covered:")
for t in afno_ds.time.values:
    print(f"  {str(t)[:16]}")

In [ ]:
# Plot all 10 AFNO timestamps in a 2×5 grid
fig, axes = plt.subplots(2, 5, figsize=(22, 8))
axes = axes.flatten()
cmap = plt.cm.Blues
vmax = max(10.0, float(afno_ds.tp.max()))

for i, t in enumerate(afno_ds.time.values):
    tp = afno_ds["tp"].sel(time=t).values
    ts_str = str(t)[:16].replace("T", "\n")
    im = axes[i].imshow(
        tp, cmap=cmap, vmin=0, vmax=vmax,
        extent=[SG_LON_MIN, SG_LON_MAX, SG_LAT_MIN, SG_LAT_MAX],
        origin="upper", aspect="auto",
    )
    axes[i].set_title(f"{ts_str}\nmax={tp.max():.2f} mm", fontsize=8)
    axes[i].set_xlabel("Lon")
    axes[i].set_ylabel("Lat")
    plt.colorbar(im, ax=axes[i], label="mm/6h", shrink=0.8)

fig.suptitle("PrecipitationAFNO (0.25°) — Singapore Hindcast (Stage 2)", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(RESULTS / "stage2_afno_all_timestamps.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → results/stage2_afno_all_timestamps.png")

In [ ]:
# Side-by-side: ERA5 (coarse) vs AFNO for the 2022-04-22 flash flood event
# ERA5 local zarr covers 2022; lat is descending (90→-90), tp in metres → convert to mm
era5 = xr.open_zarr(ERA5_PATH, consolidated=True)
era5_sg = era5.sel(
    latitude=slice(SG_LAT_MAX, SG_LAT_MIN),
    longitude=slice(SG_LON_MIN, SG_LON_MAX),
)

event_times = ["2022-04-22T00:00", "2022-04-22T06:00", "2022-04-22T12:00", "2022-04-22T18:00"]
cmap = plt.cm.Blues
vmax = 15.0  # mm/6h

fig, axes = plt.subplots(2, 4, figsize=(22, 8))

for col, ts in enumerate(event_times):
    afno_tp = afno_ds["tp"].sel(time=ts, method="nearest").values
    era5_tp = era5_sg["tp"].sel(time=ts, method="nearest").values * 1000  # m → mm

    im0 = axes[0, col].imshow(
        afno_tp, cmap=cmap, vmin=0, vmax=vmax,
        extent=[SG_LON_MIN, SG_LON_MAX, SG_LAT_MIN, SG_LAT_MAX],
        origin="upper", aspect="auto",
    )
    axes[0, col].set_title(f"AFNO  {ts[11:]}\nmax={afno_tp.max():.2f} mm", fontsize=9)
    plt.colorbar(im0, ax=axes[0, col], label="mm/6h")

    im1 = axes[1, col].imshow(
        era5_tp, cmap=cmap, vmin=0, vmax=vmax,
        extent=[SG_LON_MIN, SG_LON_MAX, SG_LAT_MIN, SG_LAT_MAX],
        origin="upper", aspect="auto",
    )
    axes[1, col].set_title(f"ERA5  {ts[11:]}\nmax={era5_tp.max():.2f} mm", fontsize=9)
    plt.colorbar(im1, ax=axes[1, col], label="mm/6h")

axes[0, 0].set_ylabel("PrecipitationAFNO (0.25°)", fontsize=9)
axes[1, 0].set_ylabel("ERA5 reanalysis (0.25°)", fontsize=9)
fig.suptitle("2022-04-22 Flash Flood Event — ERA5 vs PrecipitationAFNO", fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS / "stage2_era5_vs_afno_20220422.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → results/stage2_era5_vs_afno_20220422.png")

In [ ]:
# Summary statistics across all 10 timestamps
print("=== Stage 2 Summary Statistics ===")
print(f"Timestamps : {len(afno_ds.time.values)}")
print(f"Domain     : lat {SG_LAT_MIN}–{SG_LAT_MAX}°N, lon {SG_LON_MIN}–{SG_LON_MAX}°E")
print(f"Resolution : 0.25° (~25 km) — coarse baseline, not the 1–2 km Stage 4 target")
print()
print(f"{'Timestamp':<20} {'max (mm)':>10} {'mean (mm)':>10} {'wet frac':>10}")
print("-" * 55)
for t in afno_ds.time.values:
    tp = afno_ds["tp"].sel(time=t).values
    ts_str = str(t)[:16]
    wet_frac = (tp > 0.1).sum() / tp.size
    print(f"{ts_str:<20} {tp.max():>10.2f} {tp.mean():>10.3f} {wet_frac:>10.1%}")

era5.close()
afno_ds.close()
print("\nDone. Next: Stage 3 radar preprocessing → python scripts/preprocess_radar.py")